## **Experiment-8:**

In [1]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ==========================================
# STEP 1: LOAD REAL UCI ADULT DATASET
# ==========================================

print("Downloading UCI Adult Census Dataset...")

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.iloc[:, 0].copy()

# ==========================================
# STEP 2: CLEAN COLUMN NAMES
# ==========================================

X.columns = (
    X.columns.str.strip()
    .str.lower()
    .str.replace("-", "_")
)

# ==========================================
# STEP 3: HANDLE MISSING VALUES
# ==========================================

for col in X.select_dtypes(
    include=["object", "category", "string"]
).columns:

    X[col] = (
        X[col]
        .astype(object)
        .apply(lambda v: v.strip() if isinstance(v, str) else v)
    )

    X[col] = X[col].replace(
        ["?", ""],
        np.nan
    )

# Clean target
y = (
    y.astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
)

# Remove invalid target records
valid = y.isin(["<=50K", ">50K"])

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)

# ==========================================
# STEP 4: PRESERVE DEMOGRAPHIC ATTRIBUTES
# ==========================================

groups = X[["sex", "race"]].copy()

# Remove sensitive attributes from model input
model_X = X.drop(
    columns=["sex", "race"]
)

# ==========================================
# STEP 5: IDENTIFY FEATURE TYPES
# ==========================================

numeric_features = model_X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = model_X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nDataset Shape:", X.shape)

# ==========================================
# STEP 6: PREPROCESSING
# ==========================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# ==========================================
# STEP 7: TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test, groups_train, groups_test = (
    train_test_split(
        model_X,
        y,
        groups,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

# ==========================================
# STEP 8: CREATE ML MODEL
# ==========================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

# ==========================================
# STEP 9: TRAIN MODEL
# ==========================================

print("\nTraining Logistic Regression Model...")

model.fit(X_train, y_train)

# ==========================================
# STEP 10: MODEL EVALUATION
# ==========================================

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\nMODEL PERFORMANCE")
print("-------------------------")

print(
    "Model Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

# ==========================================
# STEP 11: DEMOGRAPHIC FAIRNESS ANALYSIS
# ==========================================

audit = groups_test.copy()

audit["Actual"] = (
    y_test.to_numpy() == ">50K"
).astype(int)

audit["Predicted"] = (
    predictions == ">50K"
).astype(int)

print("\nDEMOGRAPHIC BIAS ANALYSIS")
print("=========================")

for attribute in ["sex", "race"]:

    print("\nAnalysis by:", attribute.upper())

    result = audit.groupby(attribute).agg(
        Total_Records=("Predicted", "count"),

        Actual_Positive_Rate=("Actual", "mean"),

        Predicted_Positive_Rate=("Predicted", "mean"),

        Prediction_Accuracy=(
            "Predicted",
            lambda p: np.mean(
                p.to_numpy() ==
                audit.loc[p.index, "Actual"].to_numpy()
            )
        )
    )

    result[
        [
            "Actual_Positive_Rate",
            "Predicted_Positive_Rate",
            "Prediction_Accuracy"
        ]
    ] *= 100

    print(result.round(2))

    gap = (
        result["Predicted_Positive_Rate"].max()
        - result["Predicted_Positive_Rate"].min()
    )

    print(
        "\nPrediction Rate Difference:",
        round(gap, 2),
        "percentage points"
    )

print("\nAnalysis Completed Successfully!")


Dataset Shape: (48842, 14)

Training Logistic Regression Model...

MODEL PERFORMANCE
-------------------------
Model Accuracy: 84.55 %

DEMOGRAPHIC BIAS ANALYSIS

Analysis by: SEX
        Total_Records  Actual_Positive_Rate  Predicted_Positive_Rate  \
sex                                                                    
Female           3289                 11.16                     7.02   
Male             6480                 30.42                    24.72   

        Prediction_Accuracy  
sex                          
Female                91.61  
Male                  80.97  

Prediction Rate Difference: 17.7 percentage points

Analysis by: RACE
                    Total_Records  Actual_Positive_Rate  \
race                                                      
Amer-Indian-Eskimo            104                  8.65   
Asian-Pac-Islander            304                 25.66   
Black                         944                 12.82   
Other                          69           

c:\Users\Hp\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## **Experiment-11:**

In [2]:
import requests
import csv
from bs4 import BeautifulSoup

# Website URL
url = "https://amanbhardwaj.vercel.app/"

# Fetch website content
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Store extracted project information
projects = []
seen_urls = set()

# Define project sections
sections = {
    "Featured Projects": "projects",
    "Live Projects": "live-projects"
}

# Extract project information
for category, section_id in sections.items():

    section = soup.find("section", id=section_id)

    if section is None:
        continue

    cards = section.select(".bento-card")

    for card in cards:

        title_tag = card.find("h3")
        description_tag = card.find("p")
        project_url = card.get("href", "")

        if not title_tag or not project_url:
            continue

        title = title_tag.get_text(" ", strip=True)

        description = (
            description_tag.get_text(" ", strip=True)
            if description_tag else "Not available"
        )

        # Remove duplicate project links
        if project_url in seen_urls:
            continue

        seen_urls.add(project_url)

        projects.append([
            category,
            title,
            description,
            project_url
        ])

# Save data into CSV
filename = "portfolio_projects.csv"

with open(filename, "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Category",
        "Project Title",
        "Description",
        "Project URL"
    ])

    writer.writerows(projects)

# Display extracted records
print("======================================")
print("       PORTFOLIO DATA EXTRACTOR")
print("======================================")

for category, title, description, link in projects:

    print("\nCategory:", category)
    print("Project:", title)
    print("Description:", description)
    print("URL:", link)

# Display summary
featured = sum(
    1 for p in projects if p[0] == "Featured Projects"
)

live = sum(
    1 for p in projects if p[0] == "Live Projects"
)

print("\n======================================")
print("          EXTRACTION SUMMARY")
print("======================================")
print("Featured Projects:", featured)
print("Live Projects:", live)
print("Total Projects Extracted:", len(projects))
print("CSV File:", filename)
print("Data saved successfully!")

       PORTFOLIO DATA EXTRACTOR

Category: Featured Projects
Project: Aletheia: Epistemic Auditor
Description: Multi-agent scientific fact verification system that analyzes research manuscripts using NLI,
                    LangChain, FAISS and multiple LLMs for automated forensic claim verification.
URL: https://github.com/GitHub-AmanBhardwaj/Aletheia-EpistemicAuditor

Category: Featured Projects
Project: Multimodal Deception Detection
Description: Multimodal deep learning system combining DistilBERT, ViT & LSTM for joint text and video deception
                    analysis.
URL: https://github.com/GitHub-AmanBhardwaj/TrueLies

Category: Featured Projects
Project: GAN-Based FSR
Description: RRDBNet-based GAN framework for 4× face super-resolution, reconstructing high-resolution images from
                    low-resolution inputs.
URL: https://github.com/GitHub-AmanBhardwaj/GAN-Based-Face-Super-Resolution-RRDBNet

Category: Featured Projects
Project: Research Knowledge Assistant
De